In [ ]:
#@title 1. Install required packages and import libraries
# ── Installation ──────────────────────────────────────────────────────────────
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("EMD-signal", "PyEMD")
_ensure("plotly")
_ensure("pandas")
_ensure("numpy")

# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import re
from collections import defaultdict

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

from PyEMD import EMD

In [ ]:
#@title 2. Graphical settings (font, size, line width)
# ── Global plot style variables ───────────────────────────────────────────────
FONT_FAMILY  = "serif"
FONT_SIZE    = 14
LINE_WIDTH   = 1.5

LAYOUT_DEFAULTS = dict(
    font=dict(family=FONT_FAMILY, size=FONT_SIZE),
    legend=dict(font=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
    xaxis=dict(title_font=dict(family=FONT_FAMILY, size=FONT_SIZE),
               tickfont=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
    yaxis=dict(title_font=dict(family=FONT_FAMILY, size=FONT_SIZE),
               tickfont=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
)

print(f"Font: {FONT_FAMILY}, Size: {FONT_SIZE}, Line width: {LINE_WIDTH}")

In [ ]:
#@title 3. Retrieve data from GitHub
DATA_URL = (
    "https://raw.githubusercontent.com/postnicov/RowingSPb/"
    "refs/heads/main/data/2026_05_27_1_MH8%2BTest.csv"
)

df_raw = pd.read_csv(DATA_URL)
print(f"Loaded {len(df_raw)} rows × {len(df_raw.columns)} columns")
print("Columns:", list(df_raw.columns))
df_raw.head(3)

In [ ]:
#@title 4. Column descriptions
from IPython.display import Markdown, display

display(Markdown("""
## Column descriptions

- **T** — time, s
- **As** — boat acceleration, m/s²
- **Vs** — boat speed (GPS), m/s
- **GPSd** — distance covered (GPS), m
- **A#** — horizontal oar angle, degrees (# = rower number counted from the stern; stroke = 1)
- **V#** — vertical oar angle, degrees
- **H#** — handle force, N
- **S#** — seat position, m
"""))

In [ ]:
#@title 5. Plot As vs T (interactive Plotly)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_raw["T"], y=df_raw["As"],
    mode="lines",
    name="As",
    line=dict(width=LINE_WIDTH),
))

fig.update_layout(
    title="Raw data — As vs T",
    xaxis_title="T (s)",
    yaxis_title="As",
    **LAYOUT_DEFAULTS,
)
fig.show()

In [ ]:
#@title 6. Set time window: Tmin and Tmax
Tmin = float(df_raw["T"].min())
Tmax = float(df_raw["T"].max())

print(f"Tmin = {Tmin:.4f} s")
print(f"Tmax = {Tmax:.4f} s")

In [ ]:
#@title 6a. Retype Tmin and Tmax
Tmin = float(Tmin)  #@param {type:"number"}
Tmax = float(Tmax)  #@param {type:"number"}

print(f"Tmin = {Tmin:.4f} s")
print(f"Tmax = {Tmax:.4f} s")

In [ ]:
#@title 7. Trim to [Tmin, Tmax] and redefine T = T - Tmin
mask = (df_raw["T"] >= Tmin) & (df_raw["T"] <= Tmax)
df = df_raw.loc[mask].copy().reset_index(drop=True)
df["T"] = df["T"] - Tmin

print(f"Rows after trimming: {len(df)}")
print(f"New T range: [{df['T'].min():.4f}, {df['T'].max():.4f}] s")

In [ ]:
#@title 8. Empirical Mode Decomposition (PyEMD) for each signal column
signal_cols = [c for c in df.columns if c not in ("N", "T")]

T_arr  = df["T"].to_numpy()
emd    = EMD()
imfs   = {}
trends = {}

for col in signal_cols:
    sig = df[col].to_numpy(dtype=float)
    result = emd.emd(sig, T_arr)
    imfs[col]   = result
    trends[col] = result[-1]
    print(f"  {col:5s}: {result.shape[0]} IMFs decomposed")

print("EMD complete.")

In [ ]:
#@title 9. Plot maximal trend IMF grouped by letter prefix
groups = defaultdict(list)
for col in signal_cols:
    m = re.match(r'^([A-Za-z]+)', col)
    if m:
        groups[m.group(1)].append(col)

for letter, members in sorted(groups.items()):
    if len(members) <= 1:
        continue
    fig = go.Figure()
    for col in sorted(members):
        fig.add_trace(go.Scatter(
            x=T_arr, y=trends[col],
            mode="lines",
            name=col,
            line=dict(width=LINE_WIDTH),
        ))
    fig.update_layout(
        title=f"Trend IMF — group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title=f"{letter} (trend IMF)",
        **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
#@title 10. Detrend signals by subtracting the trend IMF
detrended = {}
for col in signal_cols:
    detrended[col] = df[col].to_numpy(dtype=float) - trends[col]

print("Detrending complete.")
for col in signal_cols[:3]:
    print(f"  {col}: mean detrended = {detrended[col].mean():.4f}")

In [ ]:
#@title 11. Plot detrended signals only for grouped variables
grouped_only = {
    letter: sorted(members)
    for letter, members in groups.items()
    if len(members) > 1
}

for letter, members in sorted(grouped_only.items()):
    fig = go.Figure()
    for col in members:
        fig.add_trace(go.Scatter(
            x=T_arr,
            y=detrended[col],
            mode="lines",
            name=col,
            line=dict(width=LINE_WIDTH),
        ))
    fig.update_layout(
        title=f"Detrended signals — group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title=f"{letter} (detrended)",
        **LAYOUT_DEFAULTS,
    )
    fig.show()